# Notebook 04: Data Cleaning and Preparation

In this notebook, we clean and prepare the datasets for further analysis 
and loading into PostgreSQL.

**Main goals:**
- Validate data types
- Handle duplicates and missing values
- Create final, clean tables
- (Optional) Save processed datasets for reuse

---

## Why clean data?

Even after splitting the raw CSV into logical tables, the data may contain:
- missing values,
- inconsistent formats,
- incorrect data types.

Cleaning at this stage ensures the dimension (customers, products)
and fact (orders) tables are accurate and database-ready.

In [ ]:
# Import necessary libraries
import pandas as pd

# Load raw dataset
df = pd.read_csv('1_data/raw/amazon_sales.csv')

# Quick check
print("Raw dataset shape:", df.shape)
df.head()


## Create logical tables

We split the raw dataset into three tables:
1. Customers
2. Products
3. Orders

This makes each table a self-contained, database-ready entity.

In [ ]:
# Orders table (fact table)
df_orders = df[[
    "OrderID", "OrderDate", "CustomerID", "ProductID",
    "Quantity", "UnitPrice", "Discount", "Tax",
    "ShippingCost", "TotalAmount","OrderStatus","PaymentMethod"
]].copy()

# Customers table
df_customers = df[[
    "CustomerID", "CustomerName", "City", "State", "Country"
]].copy()

# Products table
df_products = df[[
    "ProductID", "ProductName", "Category", "Brand"
]].copy()

## Cleaning Customers Table (Country Column)

**Issue:** Many rows have non-US countries paired with US states.

**Action:** Since each state corresponds to a US state code, 
we standardize all `Country` values to `'United States'`.

In [ ]:
# 1. Validate Country vs State consistency
print("Country distribution:")
print(df_customers["Country"].value_counts())

print("\nState distribution (top 10):")
print(df_customers["State"].value_counts().head(10))

# 2. Check non-US countries paired with US states
non_us = df_customers[df_customers["Country"] != "United States"]

print("\nNon-US countries with states (top 10):")
print(
    non_us.groupby(["Country", "State"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(10)
)

# 3. Correction: standardize Country values
df_customers["Country"] = "United States"

print("\nUpdated country distribution:")
print(df_customers["Country"].value_counts())

## Cleaning Product Table (Category Column)

**Issue:** Mismatch between `ProductName` and `Category`.

**Action:** Implemented a mapping-based correction step to align each `ProductName` with its correct `Category` before analysis.

In [ ]:
mapping = pd.read_csv("1_data/mapping/product_category_mapping.csv")

df_products = df_products.merge(mapping, on="ProductName", how="left")

df_products["Category"] = df_products["CorrectCategory"].fillna(df_products["Category"])

df_products = df_products.drop(columns=["CorrectCategory"])

In [ ]:
# Final check of products table
df_products[["ProductName","Category"]].head(10)

## Product Table Validation

We check:
- Total number of products
- Unique ProductIDs and ProductNames
- Missing values
- Category and Brand distributions

In [ ]:
# Products validation
print("Products count:", len(df_products))
print("Unique ProductIDs:", df_products["ProductID"].nunique())
print("Unique ProductNames:", df_products["ProductName"].nunique())
print("Missing values per column:")
print(df_products.isnull().sum())
print("\nCategory distribution (top 10):")
print(df_products["Category"].value_counts().head(10))
print("\nBrand distribution (top 10):")
print(df_products["Brand"].value_counts().head(10))

Products table:
- 50 unique products (IDs and names) 
- No missing values 
- Categories and brands distribution looks consistent

## Cleaning Dimension Tables

Remove duplicates and ensure that each entity 
(customer, product) is represented only once.

In [ ]:
# Customers: keep one row per CustomerID
df_customers_clean = (
    df_customers
    .drop_duplicates(subset="CustomerID")
    .reset_index(drop=True)
)

# Ensure string columns are strings
string_cols_cust = ["CustomerID", "CustomerName", "City", "State", "Country"]
for col in string_cols_cust:
    df_customers_clean[col] = df_customers_clean[col].astype(str)

# Products: keep one row per ProductID
df_products_clean = (
    df_products
    .drop_duplicates(subset="ProductID")
    .reset_index(drop=True)
)

# Ensure string columns are strings
string_cols_prod = ["ProductID", "ProductName", "Category", "Brand"]
for col in string_cols_prod:
    df_products_clean[col] = df_products_clean[col].astype(str)


## Cleaning Orders Table

Ensure correct data types for all columns:
- `OrderDate` → datetime
- Numeric columns → numeric types
- String columns → strings

In [ ]:
df_orders_clean = df_orders.copy()

# Convert OrderDate to datetime
df_orders_clean["OrderDate"] = pd.to_datetime(
    df_orders_clean["OrderDate"],
    errors="coerce"
)

# Ensure numeric columns have correct types
numeric_cols = ["Quantity", "UnitPrice", "Discount", "Tax", "ShippingCost", "TotalAmount"]
for col in numeric_cols:
    df_orders_clean[col] = pd.to_numeric(df_orders_clean[col], errors="coerce")

# Ensure string columns are strings
string_cols_orders = ["OrderID", "CustomerID", "ProductID", "PaymentMethod", "OrderStatus"]
for col in string_cols_orders:
    df_orders_clean[col] = df_orders_clean[col].astype(str)

df_orders_clean.head()

## Standardize Column Names

Convert CamelCase / PascalCase → snake_case for PostgreSQL compatibility.

In [ ]:
import re

def to_snake_case(col):
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', col)
    s2 = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1)
    return s2.lower()

df_orders_clean.columns = [to_snake_case(c) for c in df_orders_clean.columns]
df_customers_clean.columns = [to_snake_case(c) for c in df_customers_clean.columns]
df_products_clean.columns = [to_snake_case(c) for c in df_products_clean.columns]

# Quick check
print("Orders columns:", df_orders_clean.columns.tolist())
print("Customers columns:", df_customers_clean.columns.tolist())
print("Products columns:", df_products_clean.columns.tolist())

## Final cleaned datasets

Below are the final datasets that will be loaded into PostgreSQL.

In [ ]:
print("Orders:", df_orders_clean.shape)
print("Customers:", df_customers_clean.shape)
print("Products:", df_products_clean.shape)

## Save processed datasets

Saving cleaned tables allows us to reuse them without rerunning
the entire cleaning pipeline.


In [ ]:
# Create processed data directory if it doesn't exist
os.makedirs("1_data/processed", exist_ok=True)
# 
def save_csv(df, path):
    if os.path.exists(path):
        print(f"File '{path}' already exists. It will be replaced.")
    df.to_csv(path, index=False)

# Create processed datasets
save_csv(df_orders_clean, "1_data/processed/orders_clean.csv")
save_csv(df_customers_clean, "1_data/processed/customers_clean.csv")
save_csv(df_products_clean, "1_data/processed/products_clean.csv")

## Summary

- All logical tables (customers, products, orders) have been recreated from the raw CSV.
- Deduplicated where necessary to ensure proper dimension tables.
- Data types standardized and checked for missing values.
- Country column corrected to "United States" due to state-code consistency.
- Tables are now ready to be loaded into PostgreSQL for SQL analysis or visualized in Tableau.